In [136]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [137]:
!ls

data  notebooks  README.md  requirements.txt  src


In [138]:
%cd /content/drive/MyDrive/Intent-Classification-ML-Project/

/content/drive/MyDrive/Intent-Classification-ML-Project


Loading Dataset we created from RULE BASELINE

In [139]:
import pandas as pd

# Paths should match what you used in 01
features_path = "data/rule_baseline__v1+v2.parquet"

df = pd.read_parquet(features_path)

print("Loaded shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())  # first 40 cols, just to confirm

df.head()

Loaded shape: (300000, 70)

Columns:
['Login Timestamp', 'User ID', 'Round-Trip Time [ms]', 'IP Address', 'Country', 'Region', 'City', 'ASN', 'User Agent String', 'Browser Name and Version', 'OS Name and Version', 'Device Type', 'Login Successful', 'Is Attack IP', 'Is Account Takeover', 'browser', 'os', 'hour', 'dayofweek', 'is_new_device_for_user', 'is_new_ip_for_user', 'is_off_hours', 'failed_login', 'failures_last_5', 'failure_streak', 'failure_streak_capped', 'location', 'new_location_flag', 'new_asn_flag', 'device_fingerprint', 'valid_device', 'new_device_flag', 'device_change_rate', 'ts_sec', 'delta_sec', 'new_window', 'window_id', 'logins_5min', 'failure_flag', 'burst_failure_count', 'failure_rate', 'streak_reset', 'streak_id', 'failure_streak_length', 'location_freq', 'location_rarity', 'device_type_freq', 'device_type_rarity', 'asn_freq', 'asn_rarity', 'location_rarity_q', 'device_type_rarity_q', 'asn_rarity_q', 'user_offhour_rate', 'is_unusual_time_for_user', 'user_hour_std',

,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,...,rule_new_asn,rule_recent_failures,rule_risk_score,rule_risk_band,rule_risky_device_type,rule_high_risk_country,rule_risk_score_v2,rule_risk_band_v2,rule_decision_v2,pred_attack
0,2020-02-06 17:10:54.364,-9223287066183308537,541.0,84.209.76.159,no,oslo county,oslo,41164,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6...,Chrome 69.0.3497.17.19,...,0,0,0,low,0,0,0,low,ALLOW,0
1,2020-02-06 19:52:41.530,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
2,2020-02-06 20:55:19.627,-9223258649185196422,541.0,91.208.148.129,no,-,-,49310,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,...,0,0,0,low,0,0,0,low,ALLOW,0
3,2020-02-05 21:03:20.657,-9223200578825105501,541.0,79.161.56.83,no,vestfold og telemark,holmestrand,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Chrome Mobile 81.0.4044.2033,...,0,0,0,low,0,0,0,low,ALLOW,0
4,2020-02-06 19:12:29.501,-9223199305075633823,541.0,79.161.86.86,no,-,-,29695,Mozilla/5.0 (Linux; U; Android 13.0; i phone X...,Opera Mobile 52.1.2254,...,0,0,0,low,0,0,0,low,ALLOW,0


From DATASET:

Base / raw auth attributes (match Table II):

	•	Login Timestamp
	•	User ID
	•	IP Address
	•	Country, Region, City, ASN
	•	Device Type, OS Name and Version, Browser Name and Version, User Agent String
	•	Round-Trip Time [ms]
	•	Login Successful
	•	Is Attack IP
	•	Is Account Takeover
	•	plus simplified: browser, os, hour, dayofweek

These are the nodes & raw edges that KG “sees”.

Contextual/behavioral derived features (KG-style):

	•	Novelty / change:
	•	is_new_device_for_user
	•	is_new_ip_for_user
	•	new_location_flag
	•	new_asn_flag
	•	new_device_flag
	•	device_change_rate
	•	Temporal aggregates:
	•	failures_last_5
	•	failure_streak, failure_streak_capped, failure_streak_length
	•	logins_5min
	•	burst_failure_count
	•	failure_rate
	•	Time behavior:
	•	is_off_hours
	•	is_unusual_time_for_user
	•	user_offhour_rate
	•	user_hour_std, user_hour_std_q
	•	Global rarity / population stats:
	•	location_freq, location_rarity, location_rarity_q
	•	device_type_freq, device_type_rarity, device_type_rarity_q
	•	asn_freq, asn_rarity, asn_rarity_q

These are exactly what your paper calls “KG-derived contextual features” — they conceptually come from “relationships between user, device, location, ASN over time”.

Rule/policy signals (for policy layer):

	•	rule_unusual_time, rule_off_hours
	•	rule_new_device, rule_new_asn, rule_recent_failures
	•	rule_risky_device_type, rule_high_risk_country
	•	rule_risk_score, rule_risk_band
	•	rule_risk_score_v2, rule_risk_band_v2, rule_decision_v2
	•	pred_attack

These belong to your policy / rule-based side.

# We are going to use Location/ASN/DEVICE:


Because they are the strongest, most interpretable security signals you have in this dataset, and they give you maximum value for minimum complexity.

“Who is logging in, from which place, using which device, on which network, and how typical/risky is that compared to their own history and the population.”

# Creating canonical IDs for KG entities. To make it consistent for KG graph

In [140]:
import pandas as pd

df_kg = df.copy()

# Make sure timestamp is datetime and data is sorted per user + time
df_kg["Login Timestamp"] = pd.to_datetime(df_kg["Login Timestamp"])
df_kg = df_kg.sort_values(["User ID", "Login Timestamp"]).reset_index(drop=True)

# 1) location_id — reuse your existing "location" column
# (already something like "country,region,city" and normalized)
df_kg["location_id"] = df_kg["location"].fillna("unknown,unknown,unknown")

# 2) device_id — reuse your existing device_fingerprint
df_kg["device_id"] = df_kg["device_fingerprint"].fillna("unknown_device")

# 3) asn_id — from ASN
df_kg["asn_id"] = df_kg["ASN"].fillna("unknown_asn").astype(str)

df_kg[["User ID", "Login Timestamp", "location_id", "device_id", "asn_id"]].head()

,User ID,Login Timestamp,location_id,device_id,asn_id
0,-9223287066183308537,2020-02-06 17:10:54.364,"no,oslo county,oslo","desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",41164
1,-9223258649185196422,2020-02-06 19:52:41.530,"no,-,-","mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",49310
2,-9223258649185196422,2020-02-06 20:55:19.627,"no,-,-","mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",49310
3,-9223200578825105501,2020-02-05 21:03:20.657,"no,vestfold og telemark,holmestrand","mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",29695
4,-9223199305075633823,2020-02-06 19:12:29.501,"no,-,-","mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",29695


# Build KG edge tables

User–Location edges

In [141]:
user_location_edges = (
    df_kg
    .groupby(["User ID", "location_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_location_edges shape:", user_location_edges.shape)
user_location_edges.head()

user_location_edges shape: (127304, 5)


,User ID,location_id,events,first_seen,last_seen
0,-9223287066183308537,"no,oslo county,oslo",1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,"no,-,-",2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,"no,vestfold og telemark,holmestrand",1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,"no,-,-",1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,"no,oslo county,oslo",1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


User–Device edges

In [142]:
user_device_edges = (
    df_kg
    .groupby(["User ID", "device_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_device_edges shape:", user_device_edges.shape)
user_device_edges.head()

user_device_edges shape: (140066, 5)


,User ID,device_id,events,first_seen,last_seen
0,-9223287066183308537,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,"desktop,Chrome OS 12105.100.0,Chrome 72.0.3626...",1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


User–ASN edges

In [143]:
user_asn_edges = (
    df_kg
    .groupby(["User ID", "asn_id"])
    .agg(
        events=("Login Timestamp", "count"),
        first_seen=("Login Timestamp", "min"),
        last_seen=("Login Timestamp", "max"),
    )
    .reset_index()
)

print("user_asn_edges shape:", user_asn_edges.shape)
user_asn_edges.head()

user_asn_edges shape: (121840, 5)


,User ID,asn_id,events,first_seen,last_seen
0,-9223287066183308537,41164,1,2020-02-06 17:10:54.364,2020-02-06 17:10:54.364
1,-9223258649185196422,49310,2,2020-02-06 19:52:41.530,2020-02-06 20:55:19.627
2,-9223200578825105501,29695,1,2020-02-05 21:03:20.657,2020-02-05 21:03:20.657
3,-9223199305075633823,29695,1,2020-02-06 19:12:29.501,2020-02-06 19:12:29.501
4,-9223121694105191762,15659,1,2020-02-06 09:30:49.497,2020-02-06 09:30:49.497


Above 3 tables are your explicit KG views:

	•	Nodes: User ID, location_id, device_id, asn_id
 	Edges:
	•	user–location with how often and since when
	•	user–device with how often and since when
	•	user–ASN with how often and since when

  We Found:



```
	•	user_location_edges: 127k user↔location relationships
	•	user_device_edges: 140k user↔device relationships
	•	user_asn_edges: 121k user↔ASN relationships
```
user X has used ASN Y N times between first_seen and last_seen”



Explanation: Once you have these KG tables, we can:

	•	Add per-edge stats:
	•	success_rate per user–location
	•	attack_rate per user–device
	•	time-since-last-seen as a feature (recency)
	•	Add global stats (KG-level):
	•	how many users share this device/location/asn (user_count)
	•	how many attacks are associated with each node

#  Global KG stats (population-level view). Build global stats tables


This will define for each location_id or asn_id or device_id how many

  •	how many events
	•	how many unique users
	•	how many attacks
	•	attack rate

In [144]:
# We assume df_kg already exists, with:
# - Login Timestamp (datetime)
# - User ID
# - location_id
# - device_id
# - asn_id
# - Is Attack IP

# Make sure label is numeric
df_kg["Is Attack IP"] = df_kg["Is Attack IP"].astype(int)

# 3.1 Global stats for locations
location_global_stats = (
    df_kg
    .groupby("location_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

location_global_stats["attack_rate"] = (
    location_global_stats["attacks"] / location_global_stats["events"]
)

print("location_global_stats shape:", location_global_stats.shape)
location_global_stats.head()

# 3.2 Global stats for devices
device_global_stats = (
    df_kg
    .groupby("device_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

device_global_stats["attack_rate"] = (
    device_global_stats["attacks"] / device_global_stats["events"]
)

print("device_global_stats shape:", device_global_stats.shape)
device_global_stats.head()

# 3.3 Global stats for ASNs
asn_global_stats = (
    df_kg
    .groupby("asn_id")
    .agg(
        events=("Login Timestamp", "count"),
        users=("User ID", "nunique"),
        attacks=("Is Attack IP", "sum")
    )
    .reset_index()
)

asn_global_stats["attack_rate"] = (
    asn_global_stats["attacks"] / asn_global_stats["events"]
)

print("asn_global_stats shape:", asn_global_stats.shape)
asn_global_stats.head()

location_global_stats shape: (5841, 5)
device_global_stats shape: (20440, 5)
asn_global_stats shape: (2664, 5)


,asn_id,events,users,attacks,attack_rate
0,10085,6,3,0,0.0
1,10269,6,2,0,0.0
2,10778,44,11,0,0.0
3,10991,5,3,0,0.0
4,11114,2,1,0,0.0


We Found:

	•	location_global_stats: 5,841 unique locations
	•	device_global_stats: 20,440 unique device fingerprints
	•	asn_global_stats: 2,664 ASNs

For Report/Explanation. TOP RISKY ASN's

In [145]:
# Copy to avoid accidental mutation
asn_report = asn_global_stats.copy()

# Filter: at least 50 events and some attacks
asn_risky = asn_report[
    (asn_report["events"] >= 50) &
    (asn_report["attacks"] > 0)
].copy()

# Sort by attack_rate (descending) and maybe then by events
asn_risky = asn_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

# Show top 10 for reporting
asn_risky.head(10)

,asn_id,events,users,attacks,attack_rate
1455,47655,86,1,86,1.000000
1557,500021,440,238,412,0.936364
2429,57423,197,111,184,0.934010
1574,500039,1423,27,1324,0.930429
1571,500035,138,16,107,0.775362
181,138039,58,6,41,0.706897
871,265355,62,32,43,0.693548
1263,398986,3029,774,2075,0.685045
564,206948,103,22,67,0.650485
703,22612,56,1,36,0.642857


TOP RISKY COUNTRY/LOCATION

In [146]:
loc_report = location_global_stats.copy()

# Filter: at least 200 events and non-zero attacks
loc_risky = loc_report[
    (loc_report["events"] >= 200) &
    (loc_report["attacks"] > 0)
].copy()

# Sort by attack_rate descending
loc_risky = loc_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

# Top 10 risky locations
loc_risky.head(10)

,location_id,events,users,attacks,attack_rate
4683,"pl,silesia,częstochowa",337,74,324,0.961424
4905,"ro,bucuresti,bucharest",789,73,485,0.614702
5721,"us,texas,dallas",2400,608,1392,0.580000
4886,"ro,-,-",591,80,267,0.451777
5467,"us,-,-",53507,17860,17340,0.324070
5820,"vn,-,-",202,5,65,0.321782
2075,"id,bali,denpasar",663,131,156,0.235294
4676,"pl,silesia,bielsko-biala",429,86,93,0.216783
1512,"de,rheinland-pfalz,trier",241,47,34,0.141079
5130,"tr,-,-",1342,216,186,0.138599


Top risky devices

In [147]:
dev_report = device_global_stats.copy()

# Filter: at least 50 events, some attacks
dev_risky = dev_report[
    (dev_report["events"] >= 50) &
    (dev_report["attacks"] > 0)
].copy()

dev_risky = dev_risky.sort_values(
    by=["attack_rate", "events"],
    ascending=[False, False]
)

dev_risky.head(10)

,device_id,events,users,attacks,attack_rate
1216,"desktop,Mac OS X 11.6.3,Chrome 72.0.3626.115,M...",463,1,463,1.000000
4,"bot,Other ,Linkbot 1.0,ZoomBot (Linkbot 1.0 ht...",99,1,99,1.000000
18755,"mobile,iOS 14.2.1,Other ,nslookup -q=cname ob3...",123,1,122,0.991870
2066,"mobile,Android 12.0,Chrome Mobile WebView 30.0...",252,1,241,0.956349
1188,"desktop,Mac OS X 11.6.3,Chrome 64.0.3282,Mozil...",71,1,66,0.929577
4707,"mobile,Android 4.1,Chrome Mobile WebView 85.0....",56,1,42,0.750000
864,"desktop,Mac OS X 10.14.6,Chrome 72.0.3626.116,...",56,3,35,0.625000
6096,"mobile,Android 5.5.1,Chrome Mobile 81.0.4044.1...",94,32,51,0.542553
19200,"mobile,iOS 9.3.1,Chrome Mobile iOS 50.0.2661,M...",60,19,32,0.533333
14770,"mobile,iOS 11.2.6,Firefox 20.0.0.1618,Mozilla/...",63,14,33,0.523810


# Derive KG-based rarity & risk features and merge into df_kg.

Compute rarity in global stats tables

In [148]:
import numpy as np
import pandas as pd

#  Make copies so we don't accidentally overwrite the original stats
loc_stats = location_global_stats.copy()
dev_stats = device_global_stats.copy()
asn_stats = asn_global_stats.copy()

# Helper: rarity from events → higher = rarer
def compute_rarity_from_events(events_series):
    rarity_raw = 1.0 / (events_series.astype(float) + 1.0)
    # normalize to [0,1]
    return (rarity_raw - rarity_raw.min()) / (rarity_raw.max() - rarity_raw.min() + 1e-9)

# Location rarity (KG-based)
loc_stats["loc_rarity_kg"] = compute_rarity_from_events(loc_stats["events"])

# Device rarity (KG-based)
dev_stats["dev_rarity_kg"] = compute_rarity_from_events(dev_stats["events"])

# ASN rarity (KG-based)
asn_stats["asn_rarity_kg"] = compute_rarity_from_events(asn_stats["events"])

Interpretation:


```
events = how many login events used that location_id in the whole dataset.
	•	loc_rarity_kg = rarity score between 0 and 1
	•	More events → more common → lower rarity
	•	Fewer events → rarer → higher rarity
```




In [149]:
loc_stats[["events", "loc_rarity_kg"]].head()


,events,loc_rarity_kg
0,10,0.181788
1,1,1.000000
2,9,0.199970
3,1,1.000000
4,6,0.285688


	•	Locations with only 1 event get rarity 1.0 (rarest end).
	•	Locations with 6, 9, 10 events have lower rarity (~0.18–0.29), meaning:
	•	they are not super common, but clearly more common than “only seen once” locations.

In [150]:
asn_stats[["events", "asn_rarity_kg"]].head()

,events,asn_rarity_kg
0,6,0.285698
1,6,0.285698
2,44,0.044422
3,5,0.333318
4,2,0.666659


	•	ASN with 2 events → rarity ~0.67 (quite rare)
	•	ASN with 5–6 events → rarity ~0.28–0.33
	•	ASN with 44 events → rarity ~0.044 (much more common)



Quantile buckets from rarity





```
gives you discrete categories 0–3 (0 = most common, 3 = rarest).
These are good both for analysis and as model features.
```





In [151]:
# 4 quantiles: 0 (most common) → 3 (rarest)
loc_stats["loc_rarity_kg_q"] = pd.qcut(
    loc_stats["loc_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

dev_stats["dev_rarity_kg_q"] = pd.qcut(
    dev_stats["dev_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

asn_stats["asn_rarity_kg_q"] = pd.qcut(
    asn_stats["asn_rarity_kg"],
    q=4,
    labels=False,
    duplicates="drop"
).astype(int)

loc_stats[["loc_rarity_kg", "loc_rarity_kg_q"]].head()

,loc_rarity_kg,loc_rarity_kg_q
0,0.181788,0
1,1.000000,2
2,0.199970,1
3,1.000000,2
4,0.285688,1


You now have a discrete version of rarity:

	•	0 = very common
	•	1 = somewhat common
	•	2 = somewhat rare
	•	3 = rarest

Interpretation:



```
splitting rarity into buckets 0–3 (0 = lowest rarity, 3 = highest rarity):
```

	•	A location with rarity 0.1817 got bucket 0 → relatively common among all locations.
	•	A location with rarity 1.0 got bucket 2 in your sample — this just means:
	•	After quantile splitting across all 5,841 locations, that particular value fell into the 3rd quartile (label 2).
	•	(Because of ties / distribution shape, not all “1.0” values are necessarily in the very top bucket if duplicates="drop" triggered.)

Don’t overthink the exact bucket id; the important thing is:

	•	You now have a discrete version of rarity:
	•	0 = very common
	•	1 = somewhat common
	•	2 = somewhat rare
	•	3 = rarest


# Merge KG rarity & attack-rate back into df_kg

In [152]:
# First, rename attack_rate columns so they are explicit
loc_stats = loc_stats.rename(columns={"attack_rate": "loc_attack_rate_kg"})
dev_stats = dev_stats.rename(columns={"attack_rate": "dev_attack_rate_kg"})
asn_stats = asn_stats.rename(columns={"attack_rate": "asn_attack_rate_kg"})

# Merge location KG features
df_kg = df_kg.merge(
    loc_stats[["location_id", "loc_rarity_kg", "loc_rarity_kg_q", "loc_attack_rate_kg"]],
    on="location_id",
    how="left"
)

# Merge device KG features
df_kg = df_kg.merge(
    dev_stats[["device_id", "dev_rarity_kg", "dev_rarity_kg_q", "dev_attack_rate_kg"]],
    on="device_id",
    how="left"
)

# Merge ASN KG features
df_kg = df_kg.merge(
    asn_stats[["asn_id", "asn_rarity_kg", "asn_rarity_kg_q", "asn_attack_rate_kg"]],
    on="asn_id",
    how="left"
)

# Quick look at the new columns
df_kg[
    [
        "location_id", "loc_rarity_kg", "loc_rarity_kg_q", "loc_attack_rate_kg",
        "device_id", "dev_rarity_kg", "dev_rarity_kg_q", "dev_attack_rate_kg",
        "asn_id", "asn_rarity_kg", "asn_rarity_kg_q", "asn_attack_rate_kg"
    ]
].head()

,location_id,loc_rarity_kg,loc_rarity_kg_q,loc_attack_rate_kg,device_id,dev_rarity_kg,dev_rarity_kg_q,dev_attack_rate_kg,asn_id,asn_rarity_kg,asn_rarity_kg_q,asn_attack_rate_kg
0,"no,oslo county,oslo",0.000042,0,0.013573,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",0.000021,0,0.023116,41164,0.000110,0,0.004911
1,"no,-,-",0.000021,0,0.012303,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",0.000000,0,0.014447,49310,0.054032,0,0.000000
2,"no,-,-",0.000021,0,0.012303,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",0.000000,0,0.014447,49310,0.054032,0,0.000000
3,"no,vestfold og telemark,holmestrand",0.010013,0,0.000000,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",0.499968,1,0.000000,29695,0.000000,0,0.006373
4,"no,-,-",0.000021,0,0.012303,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",0.000296,0,0.026959,29695,0.000000,0,0.006373


In [153]:
df_kg[[
    "loc_rarity_kg", "loc_attack_rate_kg",
    "dev_rarity_kg", "dev_attack_rate_kg",
    "asn_rarity_kg", "asn_attack_rate_kg"
]].describe()

,loc_rarity_kg,loc_attack_rate_kg,dev_rarity_kg,dev_attack_rate_kg,asn_rarity_kg,asn_attack_rate_kg
count,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000,300000.000000
mean,0.028526,0.092173,0.094343,0.092173,0.013580,0.092173
std,0.108623,0.158851,0.202402,0.184336,0.070730,0.164200
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000021,0.000000,0.000685,0.008448,0.000000,0.004911
50%,0.000665,0.012303,0.008034,0.018582,0.000110,0.006373
75%,0.006272,0.105706,0.071369,0.053398,0.001008,0.104741
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


a) Rarity columns (*_rarity_kg)

	•	Means are very small (0.0285, 0.0943, 0.0136):

Most events use fairly common locations/devices/ASNs.

	•	75% quantile values are still very small:
	•	For locations, 75% of events have loc_rarity_kg ≤ 0.00627.
	•	For ASNs, 75% have asn_rarity_kg ≤ 0.0010.

👉 That means truly rare infrastructure is rare — exactly as expected.

b) Attack rate columns (*_attack_rate_kg)

	•	The mean = 0.092173 for all three → that matches your global attack rate ~9.2%.

This is because for each event, you’re just attaching the attack rate of its location/device/ASN.

	•	Median (50%) location_attack_rate ~ 0.0123:
	•	For 50% of events, the associated location has attack rate ~1.2% or less.
	•	75% quantile location_attack_rate ~ 0.1057:
	•	Only 25% of events are associated with locations having attack rate ≥ ~10.6%.

Similarly for ASNs and devices.

👉 Interpretation:

	•	Most events are associated with locations/devices/ASNs that historically look pretty safe.
	•	A smaller subset of events are tied to entities with much higher observed attack rates, and these will be very useful as risk signals in your intent model.

Result:

From the knowledge graph, we compute global statistics for each location, device, and ASN, including total events, unique users, and observed attack rate. We then derive KG-based rarity scores (loc_rarity_kg, dev_rarity_kg, asn_rarity_kg) and corresponding quantile buckets, and join these back to each login event. This provides a population-level context for each authentication event, indicating whether it involves common benign infrastructure or rare, high-risk infrastructure.



# KG-based features


## Entity rarity & risk (global KG stats per event) population-level behavior

	•	Location-based KG features
	•	loc_rarity_kg → normalized rarity of this location_id (0 = very common, 1 = very rare)
	•	loc_rarity_kg_q → rarity bucket (0–3)
	•	loc_attack_rate_kg → historical attack rate for this location
	•	Device-based KG features
	•	dev_rarity_kg → rarity of this device_id
	•	dev_rarity_kg_q → bucket
	•	dev_attack_rate_kg → historical attack rate for this device fingerprint
	•	ASN-based KG features
	•	asn_rarity_kg → rarity of this asn_id
	•	asn_rarity_kg_q → bucket
	•	asn_attack_rate_kg → historical attack rate for this ASN

## User–entity relationship features (per-user KG edges)

These come from comparing current event to user history (which is effectively your user–entity edges):

	•	Novelty / change:
	•	is_new_device_for_user (you derived earlier)
	•	is_new_ip_for_user
	•	new_location_flag (user appears from a new location)
	•	new_asn_flag
	•	new_device_flag
	•	device_change_rate (how often the user changes devices)


## Time-behavior & regularity (using KG-like historical memory)

Not strictly “graph edges”, but they’re computed from the same historical behavior of each user:

	•	is_off_hours
	•	is_unusual_time_for_user
	•	user_offhour_rate
	•	user_hour_std, user_hour_std_q

These capture how this login’s time compares to the user’s normal time distribution, which is stored via historical events


# Bulding KG v2

# device_user_count, asn_user_count (from global stats)

In [154]:
# Make copies to be safe
dev_stats = device_global_stats.copy()
asn_stats = asn_global_stats.copy()

# Assume these have columns: device_id / asn_id and 'users'
# If your key columns are called differently, rename them here.

dev_stats = dev_stats.rename(columns={"users": "device_user_count"})
asn_stats = asn_stats.rename(columns={"users": "asn_user_count"})

# Merge into df_kg
df_kg = df_kg.merge(
    dev_stats[["device_id", "device_user_count"]],
    on="device_id",
    how="left"
)

df_kg = df_kg.merge(
    asn_stats[["asn_id", "asn_user_count"]],
    on="asn_id",
    how="left"
)

# Fill any missing counts with 1 (seen only once)
df_kg["device_user_count"] = df_kg["device_user_count"].fillna(1).astype(int)
df_kg["asn_user_count"] = df_kg["asn_user_count"].fillna(1).astype(int)

df_kg[["device_id", "device_user_count", "asn_id", "asn_user_count"]].head()

,device_id,device_user_count,asn_id,asn_user_count
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,41164,8113
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,49310,20
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,49310,20
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,29695,44988
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,29695,44988


	There are devices used by thousands of different users (14k+, 17k+).
	•	In a synthetic dataset, that likely represents “shared infrastructure”: proxy / emulator / bot farm device signatures.
	•	device_attack_user_count shows how many distinct users on that device had at least one attack.
	•	E.g., 313 different users on the first device had attack-labelled events.

So shared_device_flag = 1 means:

“This device is heavily shared and has at least one attacking user on it.”

Shared infra risk features

device_attack_user_count and asn_attack_user_count

In [155]:
# Helper: per (device, user) whether this user ever attacked on this device
dev_user_attack = (
    df_kg
    .groupby(["device_id", "User ID"], as_index=False)["Is Attack IP"]
    .max()
)

dev_attack_users = (
    dev_user_attack
    .groupby("device_id")["Is Attack IP"]
    .sum()
    .reset_index()
    .rename(columns={"Is Attack IP": "device_attack_user_count"})
)

# Same for ASN
asn_user_attack = (
    df_kg
    .groupby(["asn_id", "User ID"], as_index=False)["Is Attack IP"]
    .max()
)

asn_attack_users = (
    asn_user_attack
    .groupby("asn_id")["Is Attack IP"]
    .sum()
    .reset_index()
    .rename(columns={"Is Attack IP": "asn_attack_user_count"})
)

# Merge into df_kg
df_kg = df_kg.merge(dev_attack_users, on="device_id", how="left")
df_kg = df_kg.merge(asn_attack_users, on="asn_id", how="left")

df_kg["device_attack_user_count"] = df_kg["device_attack_user_count"].fillna(0).astype(int)
df_kg["asn_attack_user_count"] = df_kg["asn_attack_user_count"].fillna(0).astype(int)

df_kg[["device_id", "device_user_count", "device_attack_user_count"]].head()

,device_id,device_user_count,device_attack_user_count
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,313
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,0
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,69


shared_device_flag and shared_asn_flag

In [156]:
# “shared” = used by ≥ 5 users, and
	# •	at least 1 of those users has had an attack on this entity.

SHARED_USER_THRESHOLD = 5

df_kg["shared_device_flag"] = (
    (df_kg["device_user_count"] >= SHARED_USER_THRESHOLD) &
    (df_kg["device_attack_user_count"] >= 1)
).astype(int)

df_kg["shared_asn_flag"] = (
    (df_kg["asn_user_count"] >= SHARED_USER_THRESHOLD) &
    (df_kg["asn_attack_user_count"] >= 1)
).astype(int)

df_kg[["device_id", "device_user_count", "device_attack_user_count", "shared_device_flag"]].head()

,device_id,device_user_count,device_attack_user_count,shared_device_flag
0,"desktop,Mac OS X 10.14.6,Chrome 69.0.3497.17.1...",14081,313,1
1,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221,1
2,"mobile,iOS 7.1,Android 2.3.3.2672,Mozilla/5.0 ...",17181,221,1
3,"mobile,iOS 13.4,Chrome Mobile 81.0.4044.2033,M...",2,0,0
4,"mobile,Android 13.0,Opera Mobile 52.1.2254,Moz...",3033,69,1


In [157]:
print("Attack rate by shared_device_flag:")
print(df_kg.groupby("shared_device_flag")["Is Attack IP"].mean())

print("\nAttack rate by shared_asn_flag:")
print(df_kg.groupby("shared_asn_flag")["Is Attack IP"].mean())

Attack rate by shared_device_flag:
shared_device_flag
0    0.128578
1    0.077274
Name: Is Attack IP, dtype: float64

Attack rate by shared_asn_flag:
shared_asn_flag
0    0.017670
1    0.106461
Name: Is Attack IP, dtype: float64


This is interesting:
	•	You’d expect shared malicious devices to be riskier.
	•	But in this dataset, shared devices actually have lower attack rate than non-shared ones.

Possible reasons (and exactly how you can explain it):
	•	A lot of attack traffic may be coming from less-shared, “dedicated” devices.
	•	The synthetic generator might have made “attack devices” not necessarily the most shared ones.
	•	Shared devices might be representing things like common mobile/browser signatures that many benign users also use.

This is much more aligned with intuition:
	•	When shared_asn_flag = 1:
	•	ASN is used by many users, and
	•	There is at least one attacking user on that ASN.
	•	That group has an attack rate 6× higher than the non-shared-ASN group (10.6% vs 1.8%).

👉 This is a strong infra signal:

“If you’re on an ASN that’s widely used and has known attack users, your login is considerably more risky.”

# User diversity features

 Compute per-user counts

In [158]:
user_div = (
    df_kg
    .groupby("User ID")
    .agg(
        user_device_count=("device_id", "nunique"),
        user_location_count=("location_id", "nunique"),
        user_asn_count=("asn_id", "nunique"),
    )
    .reset_index()
)

df_kg = df_kg.merge(user_div, on="User ID", how="left")

df_kg[["User ID", "user_device_count", "user_location_count", "user_asn_count"]].head()

,User ID,user_device_count,user_location_count,user_asn_count
0,-9223287066183308537,1,1,1
1,-9223258649185196422,1,1,1
2,-9223258649185196422,1,1,1
3,-9223200578825105501,1,1,1
4,-9223199305075633823,1,1,1


We Found:

	•	user_device_count → how many different devices this user has ever used
	•	user_location_count → how many different locations
	•	user_asn_count → how many different ASNs


  “Most users authenticate from a small number of devices, locations, and ASNs, while a small group exhibits high diversity. These diversity features help distinguish normal multi-device usage from potentially compromised accounts operating over many networks or geographies"

👉 How to use this:
	•	As a feature, not as a rule:
	•	The model can learn: “extremely high diversity + other signals = riskier”.
	•	As a story point in your report:
“We observe that while most users authenticate from a single device and location, a small population shows very high infrastructure diversity. Our user diversity features (user_device_count, user_location_count, user_asn_count) capture these patterns and allow the intent model to distinguish stable accounts from those operating across unusually many devices and networks.”

In [159]:
df_kg[["user_device_count", "user_location_count", "user_asn_count"]].describe()

,user_device_count,user_location_count,user_asn_count
count,300000.000000,300000.000000,300000.000000
mean,5094.421980,1546.854973,746.549467
std,6890.702112,2091.137895,1008.521730
min,1.000000,1.000000,1.000000
25%,1.000000,1.000000,1.000000
50%,1.000000,1.000000,1.000000
75%,14417.000000,4376.000000,2111.000000
max,14417.000000,4376.000000,2111.000000


Temporal infra risk (*_recent_attack_rate_7d)

In [160]:
import pandas as pd

# Ensure sorted by time and reset index to have a clean integer-based index
df_kg = df_kg.sort_values("Login Timestamp").reset_index(drop=True)

# Initialize the new column with a default value
df_kg["asn_recent_attack_rate_7d"] = 0.0

# Group by ASN
# .groups.items() returns (asn_value, list_of_original_df_kg_indices)
for asn, grp_original_indices in df_kg.groupby("asn_id").groups.items():
    # Select the subset of df_kg for the current ASN using its original df_kg indices
    # .copy() is used to prevent SettingWithCopyWarning
    grp_data = df_kg.loc[grp_original_indices, ["Login Timestamp", "Is Attack IP"]].copy()

    # Ensure this subset is sorted by timestamp for correct rolling window calculation
    grp_data = grp_data.sort_values("Login Timestamp")

    # The index of `grp_data` at this point still consists of the original integer indices from df_kg,
    # but they might be reordered due to the `sort_values` call. This is crucial for assignment.
    current_df_kg_indices_for_group = grp_data.index

    # Temporarily set 'Login Timestamp' as the index for rolling calculation
    grp_data_indexed = grp_data.set_index("Login Timestamp")

    # Calculate the rolling mean for 'Is Attack IP'
    recent_rate_series = grp_data_indexed["Is Attack IP"].rolling("7D", min_periods=1).mean()

    # Assign the calculated rates back to the main df_kg DataFrame.
    # The order of `recent_rate_series.values` matches the sorted order of `grp_data_indexed`,
    # which in turn matches the order of `current_df_kg_indices_for_group` (the reordered integer indices).
    df_kg.loc[current_df_kg_indices_for_group, "asn_recent_attack_rate_7d"] = recent_rate_series.values

# Display descriptive statistics for the newly created feature
df_kg["asn_recent_attack_rate_7d"].describe()

,asn_recent_attack_rate_7d
count,300000.000000
mean,0.093043
std,0.167153
min,0.000000
25%,0.004822
50%,0.006668
75%,0.082540
max,1.000000


Interpretation:
	•	Global mean ≈ 0.093 matches your global attack rate (~9.2%) → good sanity check.
	•	Most ASNs are quiet recently:
	•	25% of events see ASN recent rate ≤ 0.0048
	•	50% ≤ 0.0067
→ many ASNs have very low recent attack activity.
	•	Top quartile (75%) is at 0.0825 → some ASNs have ~8–10% recent attack rates.
	•	max = 1.0 → for some ASNs, in a 7-day window, every recent event was an attack (super “hot” ASN).

👉 This is a very realistic & powerful feature:

“Even if an ASN wasn’t historically bad over the whole dataset, if in the last 7 days it’s been used mostly for attacks, we treat logins from this ASN as significantly higher risk.”

We extended the Knowledge Graph features in three directions:

	1.	Shared infrastructure risk

	•	Count how many users share a device or ASN and how many of those users have attack labels.
	•	shared_asn_flag in particular identifies “bad neighborhoods”: ASNs with many users and known attackers, which show a 6× higher attack rate than regular ASNs.

	2.	User infrastructure diversity

	•	user_device_count, user_location_count, user_asn_count capture how many unique devices, locations, and ASNs each user has used.
	•	Most users are stable (only one device/location/ASN), while a small group shows extreme diversity, which can indicate special or potentially compromised accounts.

	3.	Temporal infrastructure risk

	•	asn_recent_attack_rate_7d measures how “hot” an ASN is in the last 7 days.
	•	Many ASNs are quiet, but some show concentrated recent attack activity, allowing the model to up-weight currently active threats.

# Intent Classification

We have decided to keep target label as "Is Attack IP"


Label = Is Attack IP (binary classification: 1 = malicious IP, 0 = benign)

We can later map its outputs into intent bands (legit / suspicious / malicious) using thresholds and sequence context, or even train a custom multi-class intent model. But the first advanced step is: can we predict attack vs non-attack from all our behavioral + KG features?

In [161]:
# Label
label_col = "Is Attack IP"

Choose a curated, advanced feature set

In [162]:
# Feature set (can be tweaked later, but this is a strong starting point)
feature_cols = [

    # 1. Base numeric/time context
    "Round-Trip Time [ms]",
    "hour",
    "dayofweek",

    # 2. Categorical context (we'll later encode them)
    "Country",
    "Region",
    "City",
    "Device Type",
    "browser",
    "os",
    "ASN",

    # 3. Time-of-day / behavior features
    "is_off_hours",
    "is_unusual_time_for_user",
    "user_offhour_rate",
    "user_hour_std_q",

    # 4. Failure / error behavior
    "failed_login",
    "failures_last_5",
    "failure_streak_capped",
    "logins_5min",
    "burst_failure_count",
    "failure_rate",

    # 5. Novelty / user-history (KG-style) features
    "is_new_device_for_user",
    "is_new_ip_for_user",
    "new_location_flag",
    "new_asn_flag",
    "new_device_flag",
    "device_change_rate",

    # 6. KG rarity & risk (population level)
    "loc_rarity_kg",
    "loc_attack_rate_kg",
    "dev_rarity_kg",
    "dev_attack_rate_kg",
    "asn_rarity_kg",
    "asn_attack_rate_kg",
    "asn_recent_attack_rate_7d",   # temporal infra risk

    # 7. Shared infra risk (advanced KG)
    "device_user_count",
    "device_attack_user_count",
    "shared_device_flag",
    "asn_user_count",
    "asn_attack_user_count",
    "shared_asn_flag",

    # 8. User diversity (graph node degree)
    "user_device_count",
    "user_location_count",
    "user_asn_count",
]


# Rule-based baseline columns we want to carry along for comparison
rule_cols = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
    "rule_risky_device_type",
    "rule_high_risk_country",
    "rule_risk_score_v2",
    "rule_risk_band_v2",
    "rule_decision_v2",
    # (optional) original v1 stuff if you want:
    # "rule_risk_score",
    # "rule_risk_band",
    # "pred_attack",
]

df_model will provide you clean handpicked table with features that we intent to use for training

In [163]:
# Keep only rows where label is not missing
df_model = df_kg[["Login Timestamp", "User ID", label_col] + feature_cols + rule_cols ].copy()

# Drop rows with any missing feature (simple but clean for now)
df_model = df_model.dropna().reset_index(drop=True)

print("df_model shape:", df_model.shape)
print("Columns used:")
df_model.columns.tolist()

df_model shape: (299986, 55)
Columns used:


['Login Timestamp',
 'User ID',
 'Is Attack IP',
 'Round-Trip Time [ms]',
 'hour',
 'dayofweek',
 'Country',
 'Region',
 'City',
 'Device Type',
 'browser',
 'os',
 'ASN',
 'is_off_hours',
 'is_unusual_time_for_user',
 'user_offhour_rate',
 'user_hour_std_q',
 'failed_login',
 'failures_last_5',
 'failure_streak_capped',
 'logins_5min',
 'burst_failure_count',
 'failure_rate',
 'is_new_device_for_user',
 'is_new_ip_for_user',
 'new_location_flag',
 'new_asn_flag',
 'new_device_flag',
 'device_change_rate',
 'loc_rarity_kg',
 'loc_attack_rate_kg',
 'dev_rarity_kg',
 'dev_attack_rate_kg',
 'asn_rarity_kg',
 'asn_attack_rate_kg',
 'asn_recent_attack_rate_7d',
 'device_user_count',
 'device_attack_user_count',
 'shared_device_flag',
 'asn_user_count',
 'asn_attack_user_count',
 'shared_asn_flag',
 'user_device_count',
 'user_location_count',
 'user_asn_count',
 'rule_unusual_time',
 'rule_off_hours',
 'rule_new_device',
 'rule_new_asn',
 'rule_recent_failures',
 'rule_risky_device_type',
 

Time-based train / val / test split (advanced, not random)

Model should learn on past, validate on slightly later data, and test on the most recent slice.

Simple approach:

	1.	Sort by Login Timestamp
	2.	Split by percentiles of time (e.g., 70% / 15% / 15%)

In [164]:
# Ensure sorted by time
df_model = df_model.sort_values("Login Timestamp").reset_index(drop=True)

# Compute time cutoffs for 70 / 15 / 15 split
t1 = df_model["Login Timestamp"].quantile(0.70)
t2 = df_model["Login Timestamp"].quantile(0.85)


print("Time cutoffs:")
print("Train <= ", t1)
print("Val   <= ", t2, "(and > t1)")
print("Test  >  ", t2)

# Assign split labels
def assign_split(ts):
    if ts <= t1:
        return "train"
    elif ts <= t2:
        return "val"
    else:
        return "test"

df_model["split"] = df_model["Login Timestamp"].apply(assign_split)

print(df_model["split"].value_counts())
print("\nAttack rate by split:")
print(df_model.groupby("split")[label_col].mean())


Time cutoffs:
Train <=  2020-02-06 08:00:20.449999872
Val   <=  2020-02-06 17:58:58.335000064 (and > t1)
Test  >   2020-02-06 17:58:58.335000064
split
train    209990
val       44998
test      44998
Name: count, dtype: int64

Attack rate by split:
split
test     0.085559
train    0.094638
val      0.087248
Name: Is Attack IP, dtype: float64


train    0.0946  (~9.46%)

val      0.0872  (~8.72%)

test     0.0856  (~8.56%)

Global attack rate is around 9%. Train has slightly higher attack rate than val/test, which is totally fine and realistic.


In [165]:
print("\nDate range per split:")
print(
    df_model.groupby("split")["Login Timestamp"]
    .agg(["min", "max"])
    .sort_index()
)


Date range per split:
                          min                     max
split                                                
test  2020-02-06 17:58:58.953 2020-02-07 11:17:17.158
train 2020-02-03 12:43:30.772 2020-02-06 08:00:20.052
val   2020-02-06 08:00:20.848 2020-02-06 17:58:58.129


To study How many events does each user actually have?

Can we realistically use sequences of length 10 / 20 / 50?”

Analyze per-user history length (for sequence design)

In [166]:
# How many events per user?
user_event_counts = (
    df_model
    .groupby("User ID")
    .size()
    .rename("event_count")
)

print("Number of distinct users:", user_event_counts.shape[0])
print("\nEvent count per user (summary):")
print(user_event_counts.describe())

print("\nQuantiles of event_count:")
print(user_event_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

Number of distinct users: 114802

Event count per user (summary):
count    114802.000000
mean          2.613073
std         312.787773
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max      105980.000000
Name: event_count, dtype: float64

Quantiles of event_count:
0.25    1.0
0.50    1.0
0.75    2.0
0.90    3.0
0.95    4.0
0.99    8.0
Name: event_count, dtype: float64


Analyzing the user count and thier existing events

	Most users are tiny:
	•	50% of users have only 1 login in the data.
	•	75% have ≤ 2 logins.
	•	95% have ≤ 4 logins.
	•	Only 1% of users have > 8 events, and a few users have massive histories (max ~ 106k).

  It will still shine on:

	•	bursts of failures,
	•	unusual time patterns,
	•	repeated attempts from same risky ASN / device,
	•	those handful of long-history users.

So we should choose a short-ish sequence length that covers most real histories.

In [167]:
label_col = "Is Attack IP"

df_train = df_model[df_model["split"] == "train"].copy()
df_val   = df_model[df_model["split"] == "val"].copy()
df_test  = df_model[df_model["split"] == "test"].copy()

print("Train size:", df_train.shape[0])
print("Val size:", df_val.shape[0])
print("Test size:", df_test.shape[0])

print("\nAttack rate by split:")
print(df_train[label_col].mean(), df_val[label_col].mean(), df_test[label_col].mean())

Train size: 209990
Val size: 44998
Test size: 44998

Attack rate by split:
0.09463783989713796 0.087248322147651 0.0855593581936975


In [168]:
import numpy as np

target_attack_frac = 0.19  # you can try 0.3, 0.4, or 0.5

pos = df_train[df_train[label_col] == 1]  # attacks
neg = df_train[df_train[label_col] == 0]  # normal

n_pos = len(pos)

# Solve for how many negatives we want:
# p = n_pos / (n_pos + n_neg_target)  =>  n_neg_target = n_pos * (1-p)/p
n_neg_target = int(n_pos * (1 - target_attack_frac) / target_attack_frac)

n_neg_target = min(n_neg_target, len(neg))  # don't exceed actual negatives

neg_sampled = neg.sample(n=n_neg_target, random_state=42)

df_train_bal = (
    pd.concat([pos, neg_sampled])
    .sample(frac=1, random_state=42)  # shuffle
    .reset_index(drop=True)
)

print("Balanced train size:", df_train_bal.shape[0])

print("Original train attack rate:", df_train[label_col].mean())
print("Balanced train attack rate:", df_train_bal[label_col].mean())

Balanced train size: 104594
Original train attack rate: 0.09463783989713796
Balanced train attack rate: 0.19000133850890108


Lets check the cols and. thier types

In [169]:
print("Column dtypes in df_model:\n")
print(df_model.dtypes)


Column dtypes in df_model:

Login Timestamp              datetime64[ns]
User ID                               int64
Is Attack IP                          int64
Round-Trip Time [ms]                float64
hour                                  int32
dayofweek                             int32
Country                              object
Region                               object
City                                 object
Device Type                          object
browser                              object
os                                   object
ASN                                  object
is_off_hours                          int64
is_unusual_time_for_user              int64
user_offhour_rate                   float64
user_hour_std_q                       int64
failed_login                          int64
failures_last_5                       int64
failure_streak_capped                 int64
logins_5min                           int64
burst_failure_count                   int64
fail

In [170]:
object_cols = df_model.select_dtypes(include=["object"]).columns.tolist()

print("\nObject (likely categorical/text) columns:\n", object_cols)


Object (likely categorical/text) columns:
 ['Country', 'Region', 'City', 'Device Type', 'browser', 'os', 'ASN', 'rule_risk_band_v2', 'rule_decision_v2', 'split']


Lets derive  cat_cols and num_cols

In [171]:
label_col = "Is Attack IP"
user_col = "User ID"
time_col = "Login Timestamp"

# 1. Categorical feature columns (from your dtypes, excluding 'split')
cat_cols = [
    "Country",
    "Region",
    "City",
    "Device Type",
    "browser",
    "os",
    "ASN",
]

# 2. Rule-based / meta columns that MUST NOT be used as ML features
rule_cols = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
    "rule_risky_device_type",
    "rule_high_risk_country",
    "rule_risk_score",      # v1 score (if present)
    "rule_risk_band",       # v1 band (if present)
    "rule_risk_score_v2",   # v2 score
    "rule_risk_band_v2",    # v2 band
    "rule_decision_v2",     # ALLOW/STEP_UP/BLOCK-style decision
    "pred_attack",          # any baseline predicted label, if present
]

# 3. Columns to exclude from model features
#    - keys: time, user
#    - label
#    - split indicator
#    - categorical cols (will be encoded separately if needed)
#    - rule / meta columns
exclude_for_features = (
    [time_col, user_col, label_col, "split"]
    + cat_cols
    + rule_cols
)

# 4. Numeric feature columns = everything else used for modeling
num_cols = [
    c for c in df_model.columns
    if c not in exclude_for_features
]

print("Categorical columns:", cat_cols)
print("Numeric columns (count={}):".format(len(num_cols)))
print(num_cols)

# Optional safety check: ensure no rule column slipped into num_cols
for rc in rule_cols:
    if rc in num_cols:
        print("⚠️ Warning: rule column in num_cols:", rc)

Categorical columns: ['Country', 'Region', 'City', 'Device Type', 'browser', 'os', 'ASN']
Numeric columns (count=35):
['Round-Trip Time [ms]', 'hour', 'dayofweek', 'is_off_hours', 'is_unusual_time_for_user', 'user_offhour_rate', 'user_hour_std_q', 'failed_login', 'failures_last_5', 'failure_streak_capped', 'logins_5min', 'burst_failure_count', 'failure_rate', 'is_new_device_for_user', 'is_new_ip_for_user', 'new_location_flag', 'new_asn_flag', 'new_device_flag', 'device_change_rate', 'loc_rarity_kg', 'loc_attack_rate_kg', 'dev_rarity_kg', 'dev_attack_rate_kg', 'asn_rarity_kg', 'asn_attack_rate_kg', 'asn_recent_attack_rate_7d', 'device_user_count', 'device_attack_user_count', 'shared_device_flag', 'asn_user_count', 'asn_attack_user_count', 'shared_asn_flag', 'user_device_count', 'user_location_count', 'user_asn_count']


mappings from train only and create _id columns

In [172]:
# Mask for train split
train_mask = df_model["split"] == "train"

cat_mappings = {}

for col in cat_cols:
    # Unique categories in TRAIN ONLY
    train_values = df_model.loc[train_mask, col].astype(str).unique()

    # Reserve 0 for "unknown"
    mapping = {v: i + 1 for i, v in enumerate(train_values)}
    cat_mappings[col] = mapping

    # Map full df_model using this mapping; unseen -> 0
    df_model[f"{col}_id"] = (
        df_model[col]
        .astype(str)
        .map(mapping)
        .fillna(0)
        .astype("int32")
    )

print("Created encoded ID columns:")
print([f"{c}_id" for c in cat_cols])

# Quick sanity check
df_model[["Country", "Country_id", "Device Type", "Device Type_id"]].head()

Created encoded ID columns:
['Country_id', 'Region_id', 'City_id', 'Device Type_id', 'browser_id', 'os_id', 'ASN_id']


,Country,Country_id,Device Type,Device Type_id
0,no,1,mobile,1
1,au,2,mobile,1
2,no,1,mobile,1
3,us,3,mobile,1
4,us,3,mobile,1


final feature list for modeling

In [173]:
cat_id_cols = [f"{c}_id" for c in cat_cols]

model_feature_cols = num_cols + cat_id_cols

print("Numeric feature count:", len(num_cols))
print("Categorical ID feature count:", len(cat_id_cols))
print("Total model features:", len(model_feature_cols))

# Sanity: check for NaNs in model features
print("Total NaNs in model features:",
      df_model[model_feature_cols].isna().sum().sum())

Numeric feature count: 35
Categorical ID feature count: 7
Total model features: 42
Total NaNs in model features: 0


Rebuild clean train / val / test views

In [174]:
# Just to be explicit
label_col = "Is Attack IP"
user_col = "User ID"
time_col = "Login Timestamp"

# We already created model_feature_cols earlier
# model_feature_cols = num_cols + cat_id_cols

# Clean split DataFrames
df_train = df_model[df_model["split"] == "train"].copy()
df_val   = df_model[df_model["split"] == "val"].copy()
df_test  = df_model[df_model["split"] == "test"].copy()

print("Train size:", df_train.shape[0])
print("Val size:", df_val.shape[0])
print("Test size:", df_test.shape[0])

print("\nAttack rate by split:")
for name, df_s in [("train", df_train), ("val", df_val), ("test", df_test)]:
    print(name, ":", df_s[label_col].mean())

Train size: 209990
Val size: 44998
Test size: 44998

Attack rate by split:
train : 0.09463783989713796
val : 0.087248322147651
test : 0.0855593581936975


Sort each split by (User, Time)

In [175]:
# Sort within each split by user + time
df_train = df_train.sort_values([user_col, time_col]).reset_index(drop=True)
df_val   = df_val.sort_values([user_col, time_col]).reset_index(drop=True)
df_test  = df_test.sort_values([user_col, time_col]).reset_index(drop=True)

# Quick sanity check of event counts per user in train
user_counts = df_train[user_col].value_counts()

print("Number of distinct users in train:", user_counts.shape[0])
print("\nEvent count per user in train (summary):")
print(user_counts.describe())
print("\nQuantiles of train event_count:")
print(user_counts.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

Number of distinct users in train: 81971

Event count per user in train (summary):
count    81971.000000
mean         2.561760
std        263.302274
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max      75385.000000
Name: count, dtype: float64

Quantiles of train event_count:
0.25    1.0
0.50    1.0
0.75    2.0
0.90    3.0
0.95    4.0
0.99    8.0
Name: count, dtype: float64


Build sequences for train (first pass)


Meaning: for each event, the model will look at the last 5 logins of that user (including the current one). For users with fewer than 5 events, we’ll left-pad with zeros at the start.

In [176]:
import numpy as np

SEQ_LEN = 5  # same as before

def build_sequences_for_split_with_index(df_split, feature_cols, user_col, label_col, time_col, seq_len=5):
    """
    Returns:
      X: (N, seq_len, F)
      y: (N,)
      row_idx: (N,) indices into df_split (after sorting)
    """
    df_sorted = df_split.sort_values([user_col, time_col])

    X_list = []
    y_list = []
    idx_list = []

    for uid, g in df_sorted.groupby(user_col):
        g_feats  = g[feature_cols].values
        g_labels = g[label_col].values
        g_index  = g.index.to_numpy()        # original df_split index
        T = g_feats.shape[0]

        # pad if needed
        if T < seq_len:
            pad_len = seq_len - T
            pad_feats  = np.zeros((pad_len, g_feats.shape[1]), dtype=g_feats.dtype)
            pad_labels = np.zeros(pad_len, dtype=g_labels.dtype)
            pad_index  = np.full(pad_len, -1)   # -1 for padded steps

            feats_padded  = np.vstack([pad_feats, g_feats])
            labels_padded = np.concatenate([pad_labels, g_labels])
            index_padded  = np.concatenate([pad_index, g_index])
        else:
            feats_padded  = g_feats
            labels_padded = g_labels
            index_padded  = g_index

        T_pad = feats_padded.shape[0]
        for t in range(seq_len - 1, T_pad):
            window_feats  = feats_padded[t - seq_len + 1 : t + 1]
            window_label  = labels_padded[t]
            window_rowidx = index_padded[t]     # df_split row index of "current" event

            X_list.append(window_feats)
            y_list.append(window_label)
            idx_list.append(window_rowidx)

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    row_idx = np.array(idx_list, dtype=np.int64)

    return X, y, row_idx

Train dataset sequences.

In [177]:
# X_train_seq, y_train = build_sequences_for_split(
#     df_train,
#     feature_cols=model_feature_cols,  # 42 features (35 numeric + 7 categorical IDs)
#     user_col=user_col,
#     label_col=label_col,
#     seq_len=SEQ_LEN,
# )

# print("X_train_seq shape:", X_train_seq.shape)  # (N_train_events, 5, 42)
# print("y_train shape:", y_train.shape)
# print("Train attack rate in sequences:", y_train.mean())


X_train_seq, y_train, train_row_idx = build_sequences_for_split_with_index(
    df_train,
    feature_cols=model_feature_cols,
    user_col="User ID",
    label_col="Is Attack IP",
    time_col="Login Timestamp",
    seq_len=SEQ_LEN,
)

print("X_train_seq:", X_train_seq.shape)
print("y_train:", y_train.shape)
print("train_row_idx:", train_row_idx.shape)

X_train_seq: (165606, 5, 42)
y_train: (165606,)
train_row_idx: (165606,)


Lets Visualize one sequence as a small table

In [178]:
k = 165601  # try any index from 0 to len(y_train)-1

# 5 x 42 window of features
window = X_train_seq[k]              # shape (5, 42)

# row in df_train that this sequence ends at
row_id = train_row_idx[k]
event_row = df_train.loc[row_id]

print("Sequence", k)
print("User ID       :", event_row["User ID"])
print("Current time  :", event_row["Login Timestamp"])
print("Label (attack):", y_train[k])

# Turn the 5x42 window into a nice table with feature names
seq_df = pd.DataFrame(
    window,
    columns=model_feature_cols,
)
seq_df.index = [f"step_{i+1}" for i in range(len(seq_df))]

seq_df.head()

Sequence 165601
User ID       : 9223153573126611283
Current time  : 2020-02-05 08:14:39.773000
Label (attack): 1


,Round-Trip Time [ms],hour,dayofweek,is_off_hours,is_unusual_time_for_user,user_offhour_rate,user_hour_std_q,failed_login,failures_last_5,failure_streak_capped,...,user_device_count,user_location_count,user_asn_count,Country_id,Region_id,City_id,Device Type_id,browser_id,os_id,ASN_id
step_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
step_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
step_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
step_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
step_5,541.0,8.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,3.0,18.0,160.0,1.0,2.0,1.0,4.0


Build sequences for val & test (same function)

In [179]:
# Build sequences for validation
X_val_seq, y_val, val_row_idx = build_sequences_for_split_with_index(
    df_val,
    feature_cols=model_feature_cols,  # same 42 features
    user_col=user_col,
    label_col=label_col,
    time_col=time_col,
    seq_len=SEQ_LEN,
)

print("X_val_seq shape:", X_val_seq.shape)
print("y_val shape:", y_val.shape)
print("Val attack rate in sequences:", y_val.mean())

# Build sequences for test
X_test_seq, y_test, test_row_idx = build_sequences_for_split_with_index(
    df_test,
    feature_cols=model_feature_cols,
    user_col=user_col,
    label_col=label_col,
    time_col=time_col,
    seq_len=SEQ_LEN,
)

print("X_test_seq shape:", X_test_seq.shape)
print("y_test shape:", y_test.shape)
print("Test attack rate in sequences:", y_test.mean())

X_val_seq shape: (36706, 5, 42)
y_val shape: (36706,)
Val attack rate in sequences: 0.08865035688988177
X_test_seq shape: (36710, 5, 42)
y_test shape: (36710,)
Test attack rate in sequences: 0.08828657041678017


# Compute class weights (for imbalanced training)

In [180]:
from collections import Counter

counter = Counter(y_train)
n0, n1 = counter[0], counter[1]

total = n0 + n1
p0 = n0 / total
p1 = n1 / total

# make average weight = 1, but attack gets higher weight
w0 = 0.5 / p0
w1 = 0.5 / p1

class_weights = {0: w0, 1: w1}

print("Class counts:", counter)
print("Class weights:", class_weights)

Class counts: Counter({np.int64(0): 149269, np.int64(1): 16337})
Class weights: {0: 0.5547233518011108, 1: 5.068433616943135}


First LSTM model design

In [181]:
import tensorflow as tf
from tensorflow.keras import layers, models

SEQ_LEN = X_train_seq.shape[1]
NUM_FEATURES = X_train_seq.shape[2]

model = models.Sequential([
    layers.Input(shape=(SEQ_LEN, NUM_FEATURES)),

    # Single LSTM
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.3),

    # Dense head
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 64)             │        27,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,505 (115.25 KB)

 Trainable params: 29,505 (115.25 KB)

 Non-trainable params: 0 (0.00 B)

Training loop with class weights + validation

In [ ]:
BATCH_SIZE = 512
EPOCHS = 10

history = model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_pr_auc",
            patience=3,
            mode="max",
            restore_best_weights=True,
        )
    ],
)

Epoch 1/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 14s 32ms/step - auc: 0.6883 - loss: 0.6363 - pr_auc: 0.1846 - precision: 0.1684 - recall: 0.5436 - val_auc: 0.8324 - val_loss: 0.5569 - val_pr_auc: 0.2976 - val_precision: 0.1797 - val_recall: 0.8952
Epoch 2/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - auc: 0.8027 - loss: 0.5372 - pr_auc: 0.2699 - precision: 0.2038 - recall: 0.8062 - val_auc: 0.8409 - val_loss: 0.5278 - val_pr_auc: 0.3049 - val_precision: 0.1828 - val_recall: 0.9133
Epoch 3/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - auc: 0.8130 - loss: 0.5181 - pr_auc: 0.2760 - precision: 0.2057 - recall: 0.8206 - val_auc: 0.8427 - val_loss: 0.5043 - val_pr_auc: 0.3081 - val_precision: 0.1891 - val_recall: 0.8801
Epoch 4/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 10s 30ms/step - auc: 0.8247 - loss: 0.5105 - pr_auc: 0.3021 - precision: 0.2211 - recall: 0.8013 - val_auc: 0.8448 - val_loss: 0.5022 - val_pr_auc: 0.3121 - val_precision: 0.1874 - val_recall: 0.9069
Epoch 5/10
324/324 ━━━━━━━━━━━━━━━━━━━━ 8s

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    hist = history.history

    # 1️⃣ ROC AUC
    if "auc" in hist:
        plt.figure(figsize=(5,3))
        plt.plot(hist["auc"], label="train ROC-AUC")
        plt.plot(hist["val_auc"], label="val ROC-AUC")
        plt.xlabel("Epoch")
        plt.ylabel("ROC-AUC")
        plt.title("ROC-AUC over epochs")
        plt.legend()
        plt.show()

    # 2️⃣ PR AUC (🔥 important)
    if "pr_auc" in hist:
        plt.figure(figsize=(5,3))
        plt.plot(hist["pr_auc"], label="train PR-AUC")
        plt.plot(hist["val_pr_auc"], label="val PR-AUC")
        plt.xlabel("Epoch")
        plt.ylabel("PR-AUC")
        plt.title("PR-AUC over epochs")
        plt.legend()
        plt.show()

    # 3️⃣ Loss
    plt.figure(figsize=(5,3))
    plt.plot(hist["loss"], label="train loss")
    plt.plot(hist["val_loss"], label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss over epochs")
    plt.legend()
    plt.show()

plot_history(history)

In [ ]:
# Predict probability of attack
y_val_prob = model.predict(X_val_seq).ravel()
y_test_prob = model.predict(X_test_seq).ravel()

print("Val prob range:", y_val_prob.min(), y_val_prob.max())
print("Test prob range:", y_test_prob.min(), y_test_prob.max())

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

# 1) Get precision-recall curve on VALIDATION
prec, rec, thr = precision_recall_curve(y_val, y_val_prob)

# thr has length = len(prec)-1
prec2, rec2 = prec[:-1], rec[:-1]

# 2) Compute F1 for each threshold
f1 = (2 * prec2 * rec2) / (prec2 + rec2 + 1e-12)

# 3) Choose best threshold with a recall constraint (security-friendly)
MIN_RECALL = 0.70   # change to 0.80 if you want stricter security

mask = rec2 >= MIN_RECALL
best_idx = np.argmax(np.where(mask, f1, -1))

best_thr = float(thr[best_idx])

print("=== Best threshold from VAL ===")
print("Min recall constraint:", MIN_RECALL)
print("Best threshold:", best_thr)
print("VAL Precision:", float(prec2[best_idx]))
print("VAL Recall   :", float(rec2[best_idx]))
print("VAL F1       :", float(f1[best_idx]))


In [ ]:
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

def evaluate_binary_classifier(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).ravel()

    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)

    specificity = tn / (tn + fp + 1e-12)
    fpr = 1 - specificity
    fnr = fn / (fn + tp + 1e-12)

    print(f"\nThreshold: {threshold:.4f}")
    print(f"Confusion Matrix (TN, FP, FN, TP): {tn}, {fp}, {fn}, {tp}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1-score   : {f1:.4f}")
    print(f"FPR        : {fpr:.4f}")
    print(f"FNR        : {fnr:.4f}")
    print(f"AUC        : {auc:.4f}")

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "fpr": float(fpr),
        "fnr": float(fnr),
        "auc": float(auc),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

In [ ]:
# 4) Evaluate on VAL + TEST using this threshold
print("\n=== VAL metrics at best_thr ===")
evaluate_binary_classifier(y_val, y_val_prob, threshold=best_thr)

print("\n=== TEST metrics at best_thr (chosen from VAL) ===")
evaluate_binary_classifier(y_test, y_test_prob, threshold=best_thr)

In [ ]:
"""
print("=== Validation, threshold 0.5 ===")
val_metrics_05 = evaluate_binary_classifier(y_val, y_val_prob, threshold=0.5)
"""

Try a few thresholds to understand trade-offs

In [ ]:
"""
thresholds = [0.3, 0.5, 0.7]
val_results = []

for th in thresholds:
    print(f"\n=== Validation metrics for threshold {th} ===")
    res = evaluate_binary_classifier(y_val, y_val_prob, threshold=th)
    val_results.append(res)
"""

## Threshold 0.5 (your “default” case)

Validation @ 0.5

	•	Confusion: TN=21,263, FP=12,189, FN=286, TP=2,968
	•	Accuracy: 0.6601
	•	Precision: 0.1958 (≈ 19.6%)
	•	Recall: 0.9121 (≈ 91.2%)
	•	F1: 0.3224
	•	AUC: 0.8517

  	•	For the same kind of binary “attack vs normal” decision,
the LSTM catches ~91% of attacks instead of ~59%.

	•	At the same time, when it flags something, it’s more likely to be a real attack (precision 19.6% vs 14.9%).
	•	So it’s not just “more aggressive” — it’s genuinely smarter.

If you want a strong narrative:

“Using KG + LSTM, we improved recall from ~59% to ~91%, while also increasing precision from ~15% to ~20%.”


## Threshold 0.3 (very sensitive mode)

Validation @ 0.3

	•	Confusion: TN=17,711, FP=15,741, FN=142, TP=3,112
	•	Accuracy: 0.5673
	•	Precision: 0.1651
	•	Recall: 0.9564
	•	F1: 0.2815
	•	AUC: 0.8517 (unchanged, AUC is threshold-free)

Interpretation:

	•	Model is super sensitive:
	•	Catches ~95.6% of attacks (almost everything)
	•	But raises more false alarms (precision drops to ~16.5%, accuracy goes down)
	•	This mode is like: “if we’re paranoid and don’t want to miss anything.”

Comparison to rules:

	•	Recall jumps from ~59% → 95.6%
	•	Precision still slightly better than rules (16.5% vs 14.9%), even though you’re being very aggressive
	•	Accuracy drops (56.7% vs 65.2%) because you’re flagging a lot more events


## Threshold 0.7 (strict / low-noise mode)

Validation @ 0.7

	•	Confusion: TN=29,069, FP=4,383, FN=1,131, TP=2,123
	•	Accuracy: 0.8498
	•	Precision: 0.3263
	•	Recall: 0.6524
	•	F1: 0.4350
	•	AUC: 0.8517

This one is very interesting.

“The intent-based LSTM model provides a tunable risk score.
At a moderate threshold (0.5), it catches over 90% of attacks while also improving precision compared to the rule-based baseline.
At a stricter threshold (0.7), it still detects more attacks than the rule system but more than doubles precision, significantly reducing false positives.”

⸻


Evaluate LSTM on the test set

In [ ]:
"""
print("=== TEST metrics for threshold 0.5 ===")
test_metrics_05 = evaluate_binary_classifier(y_test, y_test_prob, threshold=0.5)

print("\n=== TEST metrics for threshold 0.7 ===")
test_metrics_07 = evaluate_binary_classifier(y_test, y_test_prob, threshold=0.7)
"""

Map LSTM probabilities → intent risk bands

	•	Low: p < 0.3
	•	Medium: 0.3 ≤ p < 0.7
	•	High: p ≥ 0.7


Build the aligned test index list

In [ ]:
import numpy as np
import pandas as pd

# This cell is no longer needed to compute target_indices because test_row_idx
# is now returned directly from build_sequences_for_split_with_index.
# Keeping the cell empty or commenting out its contents.

# seq_len = 5  # the length you used for X_*_seq

# # Take only test rows, sorted by user and time
# df_test_base = (
#     df_model[df_model["split"] == "test"]
#     .sort_values(["User ID", "Login Timestamp"])
#     .reset_index()  # original index becomes column 'index'
# )

# print("df_test_base rows:", df_test_base.shape[0])

# target_indices = []

# for uid, group in df_test_base.groupby("User ID"):
#     group = group.sort_values("Login Timestamp")
#     n = len(group)
#     if n < seq_len:
#         continue
#     # For each full window [i-seq_len+1 ... i], the label is at position i
#     for i in range(seq_len - 1, n):
#         orig_idx = group.iloc[i]["index"]  # original index in df_model
#         target_indices.append(orig_idx)

# print("Number of target indices:", len(target_indices))

In [ ]:
# Use the test_row_idx returned from sequence building to build the aligned test view
df_test_view_seq = (
    df_model.loc[test_row_idx]
    .sort_values(["User ID", "Login Timestamp"])
    .reset_index(drop=True)
)

print("df_test_view_seq rows:", df_test_view_seq.shape[0])
print("y_test:", len(y_test), " y_test_prob:", len(y_test_prob))

# Implement Fusion v2 (Decision Table)

In [ ]:
import numpy as np

# Safety: check what bands we actually have
print("Unique rule_band values:", df_test_view["rule_band"].unique())
print("Unique intent_band values:", df_test_view["intent_band"].unique())

# Decision table:
# (rule_band, intent_band) -> final_band_v2
fusion_table = {
    ("low",    "low")   : "low",
    ("low",    "medium"): "medium",
    ("low",    "high")  : "medium",

    ("medium", "low")   : "medium",
    ("medium", "medium"): "medium",
    ("medium", "high")  : "high",

    # key change: when rule = high but model = low -> STEP_UP, not BLOCK
    ("high",   "low")   : "medium",
    ("high",   "medium"): "high",
    ("high",   "high")  : "high",
}

def fuse_band_v2(row):
    key = (row["rule_band"], row["intent_band"])
    # fallback just in case of unexpected values
    return fusion_table.get(key, row["rule_band"])

df_test_view["final_band_v2"] = df_test_view.apply(fuse_band_v2, axis=1)

print("Counts by final_band_v2:")
print(df_test_view["final_band_v2"].value_counts())

In [ ]:
band_to_action = {"low": "ALLOW", "medium": "STEP_UP", "high": "BLOCK"}
df_test_view["final_action_v2"] = df_test_view["final_band_v2"].map(band_to_action)

print("\nCounts by final_action_v2:")
print(df_test_view["final_action_v2"].value_counts())

print("\nAttack rate by band (rule vs intent vs final_v2):")
print(
    df_test_view.groupby("rule_band")["Is Attack IP"].mean().rename("rule_attack_rate")
    .to_frame()
    .join(
        df_test_view.groupby("intent_band")["Is Attack IP"].mean().rename("intent_attack_rate")
    )
    .join(
        df_test_view.groupby("final_band_v2")["Is Attack IP"].mean().rename("final_v2_attack_rate")
    )
)

In [ ]:
y_true = df_test_view["Is Attack IP"].astype(int).values
y_pred_block = (df_test_view["final_action_v2"] == "BLOCK").astype(int).values

TN = np.sum((y_true == 0) & (y_pred_block == 0))
FP = np.sum((y_true == 0) & (y_pred_block == 1))
FN = np.sum((y_true == 1) & (y_pred_block == 0))
TP = np.sum((y_true == 1) & (y_pred_block == 1))
total = len(y_true)

print(f"\nConfusion matrix for BLOCK vs non-BLOCK (fusion v2):")
print(f"TN (legit allowed/step_up): {TN}")
print(f"FP (legit BLOCK):           {FP}")
print(f"FN (attack not BLOCK):      {FN}")
print(f"TP (attack BLOCK):          {TP}")
print(f"Total:                      {total}")

# Basic metrics
accuracy  = (TP + TN) / total
precision = TP / (TP + FP + 1e-9)
recall    = TP / (TP + FN + 1e-9)
fpr       = FP / (FP + TN + 1e-9)
fnr       = FN / (FN + TP + 1e-9)

print("\nMetrics (fusion v2, BLOCK=attack):")
print(f"Accuracy       : {accuracy:.4f}")
print(f"Precision (PPV): {precision:.4f}")
print(f"Recall (TPR)   : {recall:.4f}")
print(f"FPR            : {fpr:.4f}")
print(f"FNR            : {fnr:.4f}")

Attach LSTM probabilities & intent bands (now it will work)

In [ ]:
t_block = np.quantile(y_val_prob, 0.995)

def prob_to_action_fixed(p):
    if p < best_thr:
        return "ALLOW"
    elif p < t_block:
        return "STEP_UP"
    else:
        return "BLOCK"

df_test_view_seq["lstm_prob"] = y_test_prob # Add this line to create the 'lstm_prob' column
df_test_view_seq["final_action"] = df_test_view_seq["lstm_prob"].apply(prob_to_action_fixed)

print(df_test_view_seq["final_action"].value_counts())
print("\nAttack rate by action:")
print(df_test_view_seq.groupby("final_action")["Is Attack IP"].mean())

In [ ]:
df_test_view = df_test_view_seq.copy()

def prob_to_intent_band(p):
    if p < 0.3:
        return "low"
    elif p < 0.7:
        return "medium"
    else:
        return "high"

df_test_view["intent_risk_band"] = df_test_view["lstm_prob"].apply(prob_to_intent_band)

Combine rule-based and intent-based risk

In [ ]:
cols_to_check = [
    "rule_unusual_time",
    "rule_off_hours",
    "rule_new_device",
    "rule_new_asn",
    "rule_recent_failures",
    "rule_risky_device_type",
    "rule_high_risk_country",
    "rule_risk_score",
    "rule_risk_band",
    "rule_risk_score_v2",
    "rule_risk_band_v2",
    "rule_decision_v2",
    "pred_attack",
]

present = [c for c in cols_to_check if c in df_model.columns]
missing = [c for c in cols_to_check if c not in df_model.columns]

print("✅ Present columns:")
print(present)

print("\n❌ Missing columns:")
print(missing)

In [ ]:
# Map bands to ordinal levels
band_to_level = {"low": 0, "medium": 1, "high": 2}

df_test_view["rule_band"] = df_test_view["rule_risk_band_v2"]
df_test_view["intent_band"] = df_test_view["intent_risk_band"]

df_test_view["rule_level"] = df_test_view["rule_band"].map(band_to_level)
df_test_view["intent_level"] = df_test_view["intent_band"].map(band_to_level)

# Conservative fusion: take the max
df_test_view["final_level"] = df_test_view[["rule_level", "intent_level"]].max(axis=1)

level_to_band = {0: "low", 1: "medium", 2: "high"}
df_test_view["final_band"] = df_test_view["final_level"].map(level_to_band)

print("Counts by final_band:")
print(df_test_view["final_band"].value_counts())

print("\nAttack rate by band (rule vs intent vs final):")
print(
    df_test_view.groupby("rule_band")["Is Attack IP"].mean().rename("rule_attack_rate")
    .to_frame()
    .join(
        df_test_view.groupby("intent_band")["Is Attack IP"].mean().rename("intent_attack_rate")
    )
    .join(
        df_test_view.groupby("final_band")["Is Attack IP"].mean().rename("final_attack_rate")
    )
)

final_band as ALLOW / STEP_UP / BLOCK:

In [ ]:
def band_to_action(band):
    if band == "low":
        return "ALLOW"
    elif band == "medium":
        return "STEP_UP"
    else:
        return "BLOCK"

df_test_view["final_action"] = df_test_view["final_band"].apply(band_to_action)
print(df_test_view["final_action"].value_counts())

1️⃣ Low-risk band (ALLOW)

	•	Total events in low band: 15,430
	•	Attack rate ≈ 6.1%

Approximate counts:

	•	Attacks in low band ≈ 15,430 × 0.0609 ≈ 940
	•	Legit (non-attacks) ≈ 15,430 − 940 ≈ 14,490

So you can describe it like:

Out of about 15.4K logins that our system marked as LOW risk (ALLOW),
roughly 14.5K are actually normal and about 940 are actually attacks that slipped through as “safe”.

So:

	•	✅ Correct in low band ≈ 14.5K
	•	❌ Missed attacks in low band ≈ 940 (these are false negatives at the “safe” level)

Medium-risk band (STEP_UP)

	•	Total events in medium band: 18,280
	•	Attack rate ≈ 9.95%

Approximate counts:

	•	Attacks in medium band ≈ 18,280 × 0.09945 ≈ 1,818
	•	Legit ≈ 18,280 − 1,818 ≈ 16,462

So in human language:

Out of 18.3K logins that our system labeled as MEDIUM risk (STEP_UP),
around 1.8K are actual attacks and 16.5K are legit users who are being challenged with extra verification.

Here:

	•	✅ Medium band is catching a big chunk of the bad guys (this is your “suspicious / grey zone”).
	•	🤷‍♂️ It also contains many legit users, but that’s acceptable because STEP_UP = challenge, not block.


3️⃣ High-risk band (BLOCK)

	•	Total events in high band: 3,000
	•	Attack rate ≈ 8.6%

Counts:

	•	Attacks in high band ≈ 3,000 × 0.086 ≈ 258
	•	Legit ≈ 3,000 − 258 ≈ 2,742

So:

Out of 3K logins that our system marked as HIGH risk (BLOCK),
only about 258 are actually attacks and ~2.7K are legit users who would be blocked (false positives).

So:

	•	✅ Correct blocks ≈ 258
	•	❌ Legit users incorrectly blocked ≈ 2,742

This is the “harsh” part of a conservative system: high-risk band is still quite noisy.